# Bidirectional Flow-Based Frame Interpolation

Synthesizes in-between frames at time `t ∈ (0,1)` using **bidirectional DIS optical flow**
+ **occlusion-aware blending** + **iterative refinement with early stopping**.

## Algorithm
```
1. Compute bidirectional multi-scale DIS flow
2. Structure tensor → variable-radius guided filter on flow
3. Iterative refinement loop (up to N passes):
   a. Scale flows to time t, warp both frames
   b. Recompute occlusion mask on current warps
   c. Compute residual flow between warped frames
   d. Early stop if ||residual|| < threshold
   e. Apply symmetric correction: F_fwd += 0.5*corr, F_bwd -= 0.5*corr
4. Adaptive motion fallback blend
```

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────
!pip install opencv-python-headless scikit-image matplotlib -q

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from google.colab import files

print(f'OpenCV {cv2.__version__} ✓')

In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────
W        = 640
H        = 480
N_FRAMES = 60
T_INTERP = 0.5
FPS_IN   = 15
FPS_OUT  = 30
OUT_PATH = 'interpolated.mp4'

# Iterative refinement config
MAX_ITERS          = 4     # max refinement passes
CONVERGENCE_THRESH = 0.75  # stop when mean correction < this (pixels)
                           # higher = fewer iters, lower = more precise
                           # 0.75 is good for fast sports motion (10fps basketball)
                           # 0.5  is good for slower footage

print(f'Interpolating at t={T_INTERP} | {FPS_IN} fps -> {FPS_OUT} fps')
print(f'Refinement: max {MAX_ITERS} iters, stop at {CONVERGENCE_THRESH}px residual')

In [ ]:
# ── Cell 4: Optical Flow + Warp Utilities (Multi-Scale) ───────────────────

def compute_flow(src_gray, dst_gray):
    """DIS optical flow src → dst. Returns (H,W,2) float32."""
    dis = cv2.DISOpticalFlow_create(cv2.DISOpticalFlow_PRESET_MEDIUM)
    s = src_gray.clip(0, 255).astype(np.uint8)
    d = dst_gray.clip(0, 255).astype(np.uint8)
    return dis.calc(s, d, None)


def compute_flow_multiscale(src_gray, dst_gray, levels=3):
    """
    Multi-scale DIS optical flow via Gaussian pyramid.
    Coarse levels handle large motion; fine levels refine details.
    """
    s = src_gray.clip(0, 255).astype(np.uint8)
    d = dst_gray.clip(0, 255).astype(np.uint8)

    src_pyr = [s]
    dst_pyr = [d]
    for _ in range(levels - 1):
        src_pyr.append(cv2.pyrDown(src_pyr[-1]))
        dst_pyr.append(cv2.pyrDown(dst_pyr[-1]))

    flow = None
    dis  = cv2.DISOpticalFlow_create(cv2.DISOpticalFlow_PRESET_MEDIUM)
    dis.setUseSpatialPropagation(True)

    for lvl in range(levels - 1, -1, -1):
        s_lvl = src_pyr[lvl]
        d_lvl = dst_pyr[lvl]
        if flow is None:
            flow = dis.calc(s_lvl, d_lvl, None)
        else:
            h, w = s_lvl.shape[:2]
            flow = cv2.resize(flow, (w, h), interpolation=cv2.INTER_LINEAR) * 2.0
            flow = dis.calc(s_lvl, d_lvl, flow)

    return flow


def warp(frame, flow):
    """
    Backward warp frame by flow using remap.
    frame : float32 (H,W,3),  flow : float32 (H,W,2)
    """
    H_, W_ = flow.shape[:2]
    gx, gy = np.meshgrid(np.arange(W_, dtype=np.float32),
                          np.arange(H_, dtype=np.float32))
    map_x = gx + flow[..., 0]
    map_y = gy + flow[..., 1]
    return np.stack([
        cv2.remap(frame[..., c], map_x, map_y,
                  interpolation=cv2.INTER_LINEAR,
                  borderMode=cv2.BORDER_REPLICATE)
        for c in range(3)
    ], axis=-1)


def warp_flow(flow, by_flow):
    """Warp a flow field by another flow — for occlusion consistency check."""
    H_, W_ = by_flow.shape[:2]
    gx, gy = np.meshgrid(np.arange(W_, dtype=np.float32),
                          np.arange(H_, dtype=np.float32))
    map_x = gx + by_flow[..., 0]
    map_y = gy + by_flow[..., 1]
    return np.stack([
        cv2.remap(flow[..., c], map_x, map_y,
                  interpolation=cv2.INTER_LINEAR,
                  borderMode=cv2.BORDER_REPLICATE)
        for c in range(2)
    ], axis=-1)


print('Flow + warp utilities (multi-scale) ready ✓')

In [ ]:
# ── Cell 5: Occlusion Mask + Structure Tensor ─────────────────────────────

def occlusion_mask(F_fwd_t, F_bwd_t, alpha=0.01):
    """
    Softmin occlusion mask.
    M=1 → trust W0 (forward warp), M=0 → trust W1 (backward warp).
    Recomputed every refinement iteration on current warps.
    """
    occ0 = np.sum((F_fwd_t + warp_flow(F_bwd_t, F_fwd_t)) ** 2, axis=-1)
    occ1 = np.sum((F_bwd_t + warp_flow(F_fwd_t, F_bwd_t)) ** 2, axis=-1)
    e0 = np.exp(-alpha * occ0)
    e1 = np.exp(-alpha * occ1)
    M  = e0 / (e0 + e1 + 1e-8)
    return M[..., np.newaxis]


def structure_tensor(gray, sigma_grad=1.0, sigma_tensor=3.0):
    """
    Per-pixel flow reliability from Structure Tensor.
    High λ2 → corner/texture → flow reliable → tight guided filter.
    Low  λ2 → flat/edge     → flow unreliable → wide guided filter.
    Returns reliability (H,W) ∈ [0,1]. Computed once — doesn't change.
    """
    g = gray.astype(np.float32) / 255.0
    Ix = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    k    = int(4 * sigma_tensor + 1) | 1
    Jxx  = cv2.GaussianBlur(Ix * Ix, (k, k), sigma_tensor)
    Jyy  = cv2.GaussianBlur(Iy * Iy, (k, k), sigma_tensor)
    Jxy  = cv2.GaussianBlur(Ix * Iy, (k, k), sigma_tensor)
    trace = Jxx + Jyy
    det   = Jxx * Jyy - Jxy * Jxy
    disc  = np.sqrt(np.maximum(trace**2 / 4.0 - det, 0.0))
    lam2  = trace / 2.0 - disc
    p99   = float(np.percentile(lam2, 99)) + 1e-6
    return np.clip(lam2 / p99, 0.0, 1.0)


print('Occlusion mask + structure tensor ready ✓')

In [ ]:
# ── Cell 6: Guided Filter + interpolate_frame_iterative ──────────────────

def guided_filter_variable_radius(guide, src, reliability,
                                   r_min=2, r_max=12, eps=0.01):
    """
    Guided filter with adaptive radius driven by structure tensor.
    High reliability → small radius (preserve sharp flow boundaries).
    Low  reliability → large radius (smooth unreliable flow).
    """
    def _gf(r):
        d       = 2 * r + 1
        mean_g  = cv2.boxFilter(guide,       -1, (d, d))
        mean_s  = cv2.boxFilter(src,         -1, (d, d))
        mean_gs = cv2.boxFilter(guide * src, -1, (d, d))
        mean_gg = cv2.boxFilter(guide ** 2,  -1, (d, d))
        cov_gs  = mean_gs - mean_g * mean_s
        var_g   = mean_gg - mean_g ** 2
        a       = cov_gs / (var_g + eps)
        b       = mean_s - a * mean_g
        return cv2.boxFilter(a, -1, (d, d)) * guide + cv2.boxFilter(b, -1, (d, d))

    out_fine   = _gf(r_min)
    out_coarse = _gf(r_max)
    return reliability * out_fine + (1.0 - reliability) * out_coarse


def refine_flow_guided_st(flow, frame_gray, reliability):
    """Apply structure-tensor-guided variable-radius guided filter to flow."""
    guide = frame_gray.astype(np.float32) / 255.0
    fx = guided_filter_variable_radius(guide, flow[..., 0], reliability)
    fy = guided_filter_variable_radius(guide, flow[..., 1], reliability)
    return np.stack([fx, fy], axis=-1)


def compute_adaptive_params(mag_fwd, mag_bwd):
    """Adaptive motion fallback params from flow magnitude statistics."""
    motion  = np.maximum(mag_fwd, mag_bwd)
    mean_m  = float(np.mean(motion))
    std_m   = float(np.std(motion))
    p95_m   = float(np.percentile(motion, 95))
    thresh  = float(np.clip(mean_m + 1.0 * std_m, 2.0, 20.0))
    cov     = std_m / (mean_m + 1e-6)
    strength = float(np.clip(cov * 0.5, 0.15, 0.65))
    return thresh, strength, {'mean': mean_m, 'std': std_m,
                              'p95': p95_m, 'thresh': thresh,
                              'strength': strength}


def interpolate_frame(frame0, frame1, t=0.5,
                      max_iters=MAX_ITERS,
                      convergence_thresh=CONVERGENCE_THRESH,
                      occ_alpha=0.01, ms_levels=3):
    """
    Synthesize intermediate frame at time t with iterative refinement.

    Refinement loop:
      - Warps both frames to time t
      - Recomputes occlusion mask on current warped frames (not stale Pass-1 mask)
      - Measures residual flow between the two warps
      - Early stops when mean residual < convergence_thresh pixels
      - Applies symmetric correction: F_fwd += 0.5*corr, F_bwd -= 0.5*corr
        (split evenly to avoid overcorrecting one side and causing oscillation)

    Structure tensor is computed ONCE — it describes image texture,
    not the flow, so it doesn't change between iterations.
    """
    f0 = frame0.astype(np.float32)
    f1 = frame1.astype(np.float32)
    g0 = cv2.cvtColor(frame0, cv2.COLOR_BGR2GRAY).astype(np.float32)
    g1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY).astype(np.float32)
    g0_u8 = g0.clip(0, 255).astype(np.uint8)
    g1_u8 = g1.clip(0, 255).astype(np.uint8)

    # ── Base flow ────────────────────────────────────────────────────────────
    F_fwd = compute_flow_multiscale(g0, g1, levels=ms_levels)
    F_bwd = compute_flow_multiscale(g1, g0, levels=ms_levels)

    # ── Structure tensor — computed once, reused each iteration ─────────────
    st0 = structure_tensor(g0_u8)
    st1 = structure_tensor(g1_u8)
    F_fwd = refine_flow_guided_st(F_fwd, g0_u8, st0)
    F_bwd = refine_flow_guided_st(F_bwd, g1_u8, st1)

    iters_run   = 0
    corrections = []   # track residual per iteration for diagnostics
    W0 = W1 = M_occ = blend = None

    # ── Iterative refinement loop ────────────────────────────────────────────
    for iteration in range(max_iters):

        # Scale flows to time t
        F_fwd_t = -t       * F_fwd
        F_bwd_t = -(1 - t) * F_bwd

        # Warp both frames to time t
        W0 = warp(f0, F_fwd_t)
        W1 = warp(f1, F_bwd_t)

        # Recompute occlusion mask on current warps (NOT the stale Pass-1 mask)
        # As flow improves each iteration, the geometry changes —
        # keeping the old mask would blend with wrong confidence weights.
        M_occ = occlusion_mask(F_fwd_t, F_bwd_t, alpha=occ_alpha)
        blend = M_occ * W0 + (1.0 - M_occ) * W1

        # Residual: flow between the two warped frames
        # At convergence W0 ≈ W1 and this goes to zero
        bW0 = cv2.cvtColor(W0.clip(0, 255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
        bW1 = cv2.cvtColor(W1.clip(0, 255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
        flow_corr = compute_flow_multiscale(bW0, bW1, levels=2)

        corr_mag = float(np.mean(np.sqrt(
            flow_corr[..., 0] ** 2 + flow_corr[..., 1] ** 2)))
        corrections.append(corr_mag)
        iters_run += 1

        # Early stopping
        if corr_mag < convergence_thresh:
            break

        # Symmetric correction — split 50/50 to avoid oscillation
        F_fwd = F_fwd + 0.5 * flow_corr
        F_bwd = F_bwd - 0.5 * flow_corr

    # ── Adaptive motion fallback (unchanged) ────────────────────────────────
    mag_fwd = np.sqrt(F_fwd[..., 0] ** 2 + F_fwd[..., 1] ** 2)
    mag_bwd = np.sqrt(F_bwd[..., 0] ** 2 + F_bwd[..., 1] ** 2)
    thresh, strength, stats = compute_adaptive_params(mag_fwd, mag_bwd)

    motion        = np.maximum(mag_fwd, mag_bwd)
    motion_smooth = cv2.GaussianBlur(motion, (15, 15), sigmaX=5.0)
    ramp          = strength / (1.0 + np.exp(-(motion_smooth - thresh)))
    M_motion      = ramp[..., np.newaxis]

    flow_result   = blend
    linear_result = (1 - t) * f0 + t * f1
    result        = (1.0 - M_motion) * flow_result + M_motion * linear_result

    return {
        'result':        result.clip(0, 255).astype(np.uint8),
        'W0':            W0.clip(0, 255).astype(np.uint8),
        'W1':            W1.clip(0, 255).astype(np.uint8),
        'mask_occ':      M_occ,
        'mask_motion':   M_motion,
        'motion_stats':  stats,
        'struct_tensor': 0.5 * (st0 + st1),
        'iters_run':     iters_run,
        'corrections':   corrections,
        'F_fwd':         F_fwd,
        'F_bwd':         F_bwd,
    }


print('interpolate_frame() with iterative refinement + early stopping ready ✓')

In [ ]:
# ── Cell 7: Convergence Diagnostics (run on a sample pair first) ──────────
# Use this to tune CONVERGENCE_THRESH for your specific video.
# Plot shows residual magnitude per iteration across multiple frame pairs.

def plot_convergence(frames_list, t=0.5, max_iters=6, title='Convergence'):
    """
    frames_list: list of (frame0, frame1) pairs to test.
    Plots correction magnitude per iteration for each pair.
    Use to find where the curve flattens → set CONVERGENCE_THRESH there.
    """
    fig, ax = plt.subplots(figsize=(8, 4))
    for i, (f0, f1) in enumerate(frames_list):
        out = interpolate_frame(f0, f1, t=t, max_iters=max_iters,
                                convergence_thresh=0.0)  # 0 = run all iters
        ax.plot(range(1, len(out['corrections']) + 1),
                out['corrections'], marker='o', label=f'Pair {i}')
    ax.axhline(y=CONVERGENCE_THRESH, color='red', linestyle='--',
               label=f'Current threshold ({CONVERGENCE_THRESH}px)')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Mean residual (pixels)')
    ax.set_title(title); ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()

print('plot_convergence() ready ✓')
print('Call it after uploading your video: plot_convergence([(raw_frames[i], raw_frames[i+1]) for i in range(0,5)])')

In [ ]:
# ── Cell 8: Upload video → interpolate → download ─────────────────────────

import os

# ── Step 1: Upload ──────────────────────────────────────────────────────────
print('Upload your video...')
uploaded = files.upload()
input_path = list(uploaded.keys())[0]
print(f'Uploaded: {input_path}')

# ── Step 2: Read all frames with OpenCV ─────────────────────────────────────
cap = cv2.VideoCapture(input_path)
FPS_REAL_IN  = cap.get(cv2.CAP_PROP_FPS)
FPS_REAL_OUT = 60    # <-- set your target fps here
INTERP_STEPS = round(FPS_REAL_OUT / FPS_REAL_IN)

W_r = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_r = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

raw_frames = []
while True:
    ret, frame = cap.read()
    if not ret: break
    raw_frames.append(frame)
cap.release()

print(f'Input : {len(raw_frames)} frames @ {FPS_REAL_IN:.1f} fps  ({W_r}x{H_r})')
print(f'Output: {FPS_REAL_OUT} fps  ({INTERP_STEPS}x — {INTERP_STEPS-1} frames inserted per pair)')
print(f'Refinement: max {MAX_ITERS} iters, threshold {CONVERGENCE_THRESH}px')

# ── Step 3: Optional — plot convergence on first 5 pairs to tune threshold ──
# Uncomment to run before full interpolation:
# plot_convergence([(raw_frames[i], raw_frames[i+1]) for i in range(min(5, len(raw_frames)-1))])

# ── Step 4: Interpolate all frames ──────────────────────────────────────────
upsampled_frames = []
all_iters = []   # track iterations used per frame pair

for f in range(len(raw_frames) - 1):
    f0 = raw_frames[f]
    f1 = raw_frames[f + 1]
    upsampled_frames.append(f0)

    for step in range(1, INTERP_STEPS):
        t = step / INTERP_STEPS
        out = interpolate_frame(f0, f1, t=t)
        upsampled_frames.append(out['result'])
        all_iters.append(out['iters_run'])

    if (f + 1) % 10 == 0:
        avg_iters = np.mean(all_iters[-10*INTERP_STEPS:]) if all_iters else 0
        print(f'  {f+1}/{len(raw_frames)-1} pairs | avg iters: {avg_iters:.1f}')

upsampled_frames.append(raw_frames[-1])

avg_iters_total = float(np.mean(all_iters)) if all_iters else 0
print(f'Generated {len(upsampled_frames)} frames @ {FPS_REAL_OUT} fps')
print(f'Average refinement iterations used: {avg_iters_total:.2f} / {MAX_ITERS}')

# ── Step 5: Plot iteration usage histogram ───────────────────────────────────
if all_iters:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(all_iters, bins=range(1, MAX_ITERS + 2), align='left', rwidth=0.8)
    ax.set_xlabel('Iterations used'); ax.set_ylabel('Count')
    ax.set_title(f'Refinement iterations (avg={avg_iters_total:.2f})')
    ax.axvline(x=avg_iters_total, color='red', linestyle='--', label='mean')
    ax.legend(); plt.tight_layout(); plt.show()
    early_stopped_pct = 100 * sum(i < MAX_ITERS for i in all_iters) / len(all_iters)
    print(f'Early stopped: {early_stopped_pct:.1f}% of frame pairs')

# ── Step 6: Build side-by-side comparison video ──────────────────────────────
cmp_out  = 'realvideo_comparison.mp4'
fourcc_r = cv2.VideoWriter_fourcc(*'mp4v')
writer   = cv2.VideoWriter(cmp_out, fourcc_r, FPS_REAL_OUT, (W_r * 2, H_r))

orig_repeated = []
for f in range(len(raw_frames) - 1):
    for _ in range(INTERP_STEPS):
        orig_repeated.append(raw_frames[f])
orig_repeated.append(raw_frames[-1])

def put_label(img, line1, line2, color):
    cv2.putText(img, line1, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 3)
    cv2.putText(img, line1, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color,   1)
    cv2.putText(img, line2, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 3)
    cv2.putText(img, line2, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color,   1)

for i in range(len(upsampled_frames)):
    orig   = orig_repeated[i].copy()
    interp = upsampled_frames[i].copy()
    put_label(orig,   f'Original ({FPS_REAL_IN:.0f} fps)',  'No interpolation',   (180,180,180))
    put_label(interp, f'Interpolated ({FPS_REAL_OUT} fps)', 'Iterative Refinement', (100,255,150))
    panel = np.hstack([orig, interp])
    cv2.line(panel, (W_r, 0), (W_r, H_r), (200,200,200), 2)
    writer.write(panel)

writer.release()
print(f'Saved {cmp_out}')
files.download(cmp_out)
print('Done ✓')